# Sentinel-1 LULC: full run on SRH Data Lab (setup, export, train, evaluate)
Run top to bottom with the **Python 3** kernel. Everything is resumable: after a disconnect, rerun cells 1 to 3, then the cell you were on.

- Needs a **GPU session** for training (cell 1 prints `GPU: True`). Without a GPU, only exports and evaluation of finished runs make sense.
- Data and results are saved under `/home/jovyan/vault/lulc` (`raw` and `runs`). Do not delete it: finished variants are skipped, so nothing retrains.
- To reuse data already exported on Google Drive: download the `lulc/raw` and `lulc/runs` folders from Drive, upload them into `/home/jovyan/vault/lulc/` and skip the export cells.
- Edit `PROJECT` in cell 2.


In [ ]:
# 1. Unzip (put sar-lulc.zip in /home/jovyan/vault first) and install
import os
os.chdir('/home/jovyan/vault')
!unzip -qo sar-lulc.zip
%cd sar-lulc
!pip -q install -r requirements.txt
import torch; print(torch.__version__, 'GPU:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('NO GPU: exports and evaluation of finished runs are fine here, but do not start training on a CPU.')

In [4]:
# 1. Use the existing repo (no unzip, nothing overwritten) and check the GPU
import os
os.chdir('/home/jovyan/vault/sar-lulc')
import torch; print(torch.__version__, 'GPU:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('NO GPU: exports and evaluation of finished runs are fine here, but do not start training on a CPU.')

2.15.0+cu132 GPU: False
NO GPU: exports and evaluation of finished runs are fine here, but do not start training on a CPU.


In [3]:
!pip -q install -r requirements.txt

In [5]:
# 2. Settings and folders (Data Lab: local disk, no Google Drive)
PROJECT = 'sar-lulc'   # <- edit
BASE = '/home/jovyan/vault/lulc'
OUT  = f'{BASE}/raw'
RUNS = f'{BASE}/runs'
!mkdir -p $OUT $RUNS
!df -h /home/jovyan/vault | tail -1     # check free space: the full 2021 export is about 5 GB

/dev/sdb1       4.0T  2.5T  1.5T  64% /home/jovyan/vault


In [ ]:
# 3. Earth Engine login (prints a link; open it, sign in, paste the code back here)
import ee
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(auth_mode='notebook')
    ee.Initialize(project=PROJECT)

In [ ]:
# 4. Check the plan, no EE calls
%run scripts/export_dataset.py --out $OUT --dry-run

## Export (about 15 s per city-month; rural tiles faster)

In [ ]:
# 5a. Training cities
TRAIN_CITIES = 'munich berlin hamburg cologne paris amsterdam warsaw rome madrid vienna'
%run scripts/export_dataset.py --project $PROJECT --out $OUT --regions $TRAIN_CITIES

In [ ]:
# 5b. Rural train tiles and validation regions
VAL_AND_RURAL = 'poltava hungarian_plain black_forest sahara_algeria dehesa_spain finnish_lakeland stuttgart lyon beauce sweden_taiga atacama crete'
%run scripts/export_dataset.py --project $PROJECT --out $OUT --regions $VAL_AND_RURAL

In [ ]:
# 5c. Test, out-of-distribution and case study
TEST = 'frankfurt prague milan valencia north_china_plain appalachians western_desert great_basin lake_balaton heidelberg'
%run scripts/export_dataset.py --project $PROJECT --out $OUT --regions $TEST

In [ ]:
# 5d. Reference maps (WorldCover + Esri) for test regions
%run scripts/export_reference.py --project $PROJECT --out $OUT --regions $TEST

In [ ]:
# 5f. OPTIONAL, after the first full result: five new train tiles (southern hemisphere, steppe, desert)
# Scene and label coverage per tile for 2021 (no export). Drop any tile that fails.
NEW_TILES = 'pampas nsw_pasture kenya_savanna mongolia_steppe arabia_desert'
# South Africa: TEST ONLY (test_ood). Never trained on; answers 'does it work in southern Africa'.
ZA_TILES = 'za_freestate_maize za_lowveld_savanna za_karoo_shrub za_capetown'
%run scripts/ee_year_stats.py --project $PROJECT --years 2021 --regions $NEW_TILES
%run scripts/ee_year_stats.py --project $PROJECT --years 2021 --regions $ZA_TILES

In [ ]:
# 5g. Export the new tiles that passed the check (edit NEW_TILES first if one failed)
%run scripts/export_dataset.py --project $PROJECT --out $OUT --regions $NEW_TILES
# South Africa test tiles (edit ZA_TILES first if one failed the check)
%run scripts/export_dataset.py --project $PROJECT --out $OUT --regions $ZA_TILES

In [ ]:
# 5h. Extra rural TEST sites to pass 50 total (test_ood only, no retraining). Coverage check first, then export the survivors.
EXTRA_TEST = 'india_punjab deccan_india java_rice mekong_delta amazon_para cerrado_brazil canada_boreal australia_outback altiplano_andes sahel_niger iowa_maize congo_forest patagonia_steppe nz_pasture scottish_highlands iceland_tundra kazakh_steppe tibetan_plateau thai_paddy ethiopia_highlands madagascar_plateau california_valley texas_rangeland pantanal_wetland siberia_taiga bangladesh_delta'
%run scripts/ee_year_stats.py --project $PROJECT --years 2021 --regions $EXTRA_TEST
# Drop any tile that failed the check from EXTRA_TEST, then export:
%run scripts/export_dataset.py --project $PROJECT --out $OUT --regions $EXTRA_TEST

In [ ]:
# 5e. OPTIONAL, for variant V4 only: coherence for train and val regions
# ALL_TV = TRAIN_CITIES + ' ' + VAL_AND_RURAL
# %run scripts/export_coherence.py --project $PROJECT --out $OUT --regions $ALL_TV

## Training
First a 2-epoch timing check. Note seconds per epoch, then run the full thing.

In [ ]:
# 6. Timing check (separate output folder)
%run scripts/train_all.py --data $OUT --out $BASE/runs_smoke --variants V0_scratch_1m --seeds 0 --epochs 2


In [ ]:
# 7a. V0 and V1: scratch vs ImageNet, 3 seeds
%run scripts/train_all.py --data $OUT --out $RUNS --variants V0_scratch_1m V1_imagenet_1m

In [ ]:
# 7b. V2-V4 (encoder weights chosen from V0/V1). Remove V4_window3_coh50 if coherence was not exported.
%run scripts/train_all.py --data $OUT --out $RUNS --variants V2_window3 V3_window3_conf50 V4_window3_coh50

In [ ]:
# 7c. Improvements, compared with V2 on the OLD regions only (new tiles excluded), validation mIoU decides.
# V5 yearly statistic channels, V6 loss settings, V7 speckle augmentation, V8 all three,
# V11/V12: rural focus. Training patches are drawn so every region counts equally (region_balance: equal), because
# the 10 big cities otherwise supply about 90% of the samples. V12 adds the yearly statistics.
# V9 regularisation bundle from repo B (dB shift, month dropout, window jitter, label smoothing, weight decay 0.05).
EXCL = 'pampas nsw_pasture kenya_savanna mongolia_steppe arabia_desert'
%run scripts/train_all.py --data $OUT --out $RUNS --variants V5_window3_stats V6_window3_loss V7_window3_speckle V8_window3_all V9_window3_regularised V11_window3_balanced V12_window3_balanced_stats --exclude $EXCL


In [ ]:
# 7d. OPTIONAL: effect of the new tiles. Retrain the winner on ALL regions in a separate folder and compare with 7b/7c on validation.
# Example for V2; replace with the winning variant.
%run scripts/train_all.py --data $OUT --out $RUNS_newtiles --variants V2_window3


In [ ]:
# 7e. OPTIONAL: label agreement filter (V10, from reference implementations A and B).
# Needs the WorldCover/Esri reference maps of the TRAIN regions too (about 10 s per region), then trains V10 on the old regions.
# After this, WorldCover is no longer independent for V10: report Esri as the independent check.
TRAIN_REGS = 'munich berlin hamburg cologne paris amsterdam warsaw rome madrid vienna poltava hungarian_plain black_forest sahara_algeria dehesa_spain finnish_lakeland'
%run scripts/export_reference.py --project $PROJECT --out $OUT --regions $TRAIN_REGS
%run scripts/train_all.py --data $OUT --out $RUNS --variants V10_window3_agree --exclude $EXCL


In [ ]:
# 7f. OPTIONAL research variant: HyP3 coherence (V13) on the RURAL tiles only. Needs a free NASA Earthdata login (~/.netrc).
# Cost per repo A's notes: 1 credit per pair, about 60 pairs per 10 km tile. 'plan' costs nothing. Skip this if time is short.
!pip install -q asf_search hyp3_sdk rasterio
# plan run on 1 Nov, year 2021: sahara_algeria, mongolia_steppe and arabia_desert returned 0 bursts, so they are left out (V2 and V13 must use the SAME tiles).
COH_TILES = 'poltava hungarian_plain black_forest dehesa_spain finnish_lakeland pampas nsw_pasture kenya_savanna beauce sweden_taiga atacama crete'
!python scripts/hyp3_coherence.py plan --data $OUT --regions $COH_TILES
# Then, only if the plan looks right:
#   !python scripts/hyp3_coherence.py submit  --data $OUT --regions $COH_TILES --max-pairs 1500
#   !python scripts/hyp3_coherence.py collect --data $OUT --regions $COH_TILES      # repeat every few hours until all are done
# Then train V13 and the matching V2 baseline on the same tiles (separate output folder, copy V0/V1 summaries in first):
#   %run scripts/train_all.py --data $OUT --out $RUNS_coh --variants V2_window3 V13_window3_coh_hyp3 --only $COH_TILES


In [ ]:
# 8. Curves and validation table (gaps under 0.05 mIoU are seed noise)
import pandas as pd, matplotlib.pyplot as plt, glob
fig,ax=plt.subplots(1,2,figsize=(11,4))
for d in sorted(glob.glob(f'{RUNS}/V*/seed*')):
    h=pd.read_csv(f'{d}/history.csv'); v=d.split('/')[-2]
    ax[0].plot(h.epoch,h.train_loss,alpha=.5,label=v if d.endswith('seed0') else None)
    ax[1].plot(h.epoch,h.val_miou,alpha=.5,label=v if d.endswith('seed0') else None)
ax[0].set_title('train loss'); ax[1].set_title('validation mIoU (EMA weights)')
for a in ax: a.set_xlabel('epoch'); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{RUNS}/curves.png',dpi=150); plt.show()
pd.read_csv(f'{RUNS}/val_summary.csv').round(3)

## Evaluation
Choose `BEST` from the validation table, never from test numbers.

In [ ]:
# 9. Test evaluation (writes metrics.json, per_class.csv with IoU/F1/precision/recall, per_city.csv, per_climate.csv, change.csv)
BEST = 'V2_window3'   # <- set from the validation table (V5-V13 may beat it; follow the decision rules in docs/REFERENCES_ROUND2.md)
%run scripts/evaluate.py --data $OUT --runs $RUNS --variant $BEST


In [ ]:
# 9a. OPTIONAL: smoothing the probabilities instead of voting on classes (idea from repo B). Same model, separate folder.
# Compare the dw_majority and dw_probmean rows in metrics.json and the 'model+probmean3' false-change row. Uses about 1 GB extra per 24 km city.
%run scripts/evaluate.py --data $OUT --runs $RUNS --variant $BEST --prob-smooth --tag _prob


In [ ]:
# 9b. Pack the chosen model into ONE file for the prototype and the hand-in (all seeds + model cards + area-warning reference)
!python scripts/package_model.py --runs $RUNS --variant $BEST --out weights
!ls -la weights


In [ ]:
# 10. Per-class scores, per-city table, false-change table
import json
m=json.load(open(f'{RUNS}/eval_{BEST}/metrics.json'))
for k,v in m.items():
    print(k,'IoU',{c:round(x,3) for c,x in v['per_class_iou'].items()},'F1',{c:round(x,3) for c,x in v['per_class_f1'].items()})
display(pd.read_csv(f'{RUNS}/eval_{BEST}/per_city.csv').round(3))
pd.read_csv(f'{RUNS}/eval_{BEST}/change.csv').groupby('maps')[['flip_rate_pct','monthly_churn_pp','net_total_change_pp']].mean().round(2)

In [ ]:
# 11. OPTIONAL out-of-year test: export test cities for 2024, then score the 2021 model
OUT24 = f'{BASE}/raw2024'
TEST_CITIES = 'frankfurt prague milan valencia'
# %run scripts/export_dataset.py --project $PROJECT --out $OUT24 --year 2024 --regions $TEST_CITIES
# !cp $OUT/frankfurt/reference.npz $OUT24/frankfurt/   # repeat per city
# %run scripts/evaluate.py --data $OUT24 --runs $RUNS --variant $BEST --year 2024 --splits test --tag _2024
